## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pathlib

In [ ]:
data_dir = pathlib.Path("../data/")

## Merge Dataframes

In [ ]:
merge_method = "inner"

In [ ]:
def ignore(x):
    return x

In [ ]:
order_items_df = pd.read_csv(data_dir.joinpath("olist_order_items_dataset.csv"))
payments_df = pd.read_csv(data_dir.joinpath("olist_order_payments_dataset.csv"))

In [ ]:
products_df = pd.read_csv(data_dir.joinpath("olist_products_dataset.csv"))
products_df.sample()

In [ ]:
product_id_to_name = (
    products_df[["product_id", "product_category_name"]]
    .set_index("product_id")
    .to_dict()["product_category_name"]
)

In [ ]:
order_items_df["product_id"] = order_items_df["product_id"].map(product_id_to_name)
order_items_df.rename(columns={"product_id": "product_category"}, inplace=True)

In [ ]:
order_items_df.sample(5)

In [ ]:
products = order_items_df.groupby(["product_category"])["price"].mean()
mean_product_price = products.values.mean()
order_items_df["price_category"] = order_items_df["product_category"].apply(
    lambda x: "More Expensive"
    if products.get(x, np.nan) > mean_product_price
    else "Cheaper"
)

In [ ]:
aggregation_dict = {
    "order_item_id": tuple,
    "product_category": tuple,
    "seller_id": tuple,
    "shipping_limit_date": tuple,
    "price": tuple,
    "freight_value": tuple,
    "price_category": tuple,
}

order_items_df = (
    order_items_df.groupby(["order_id"]).aggregate(aggregation_dict).reset_index()
)

In [ ]:
order_items_df["total_price"] = order_items_df["price"].apply(sum)
order_items_df["total_freight"] = order_items_df["freight_value"].apply(sum)
order_items_df["full_price"] = (
    order_items_df["total_price"] + order_items_df["total_freight"]
)

In [ ]:
aggregation_dict = {
    "payment_sequential": max,
    "payment_type": tuple,
    "payment_installments": sum,
    "payment_value": sum,
}
payments_df = payments_df.groupby("order_id").aggregate(aggregation_dict).reset_index()

In [ ]:
order_items_payments_df = pd.merge(
    order_items_df, payments_df, on="order_id", how=merge_method
)
order_items_payments_df.sample(2)

In [ ]:
orders_df = pd.read_csv(data_dir.joinpath("olist_orders_dataset.csv"))
orders_payments_df = pd.merge(
    order_items_payments_df, orders_df, on="order_id", how=merge_method
)
orders_payments_df.sample()

In [ ]:
customer_df = pd.read_csv(data_dir.joinpath("olist_customers_dataset.csv"))

customer_orders_df = pd.merge(
    orders_payments_df, customer_df, on="customer_id", how=merge_method
)
customer_orders_df.sample()

In [ ]:
order_review_df = pd.read_csv(data_dir.joinpath("olist_order_reviews_dataset.csv"))
order_review_df.sample()

In [ ]:
order_review_df = order_review_df.groupby(["order_id"]).aggregate(tuple).reset_index()

In [ ]:
customer_orders_reviews = pd.merge(
    customer_orders_df, order_review_df, on="order_id", how=merge_method
)
customer_orders_reviews.sample()

In [ ]:
customer_orders_reviews.groupby(["customer_state"])["full_price"].aggregate(
    ["mean"]
).sort_values(by="mean", ascending=False).plot(kind="bar")

In [ ]:
customer_orders_reviews.sample(4)

## Charts

### Installment Use by Product Category (Frequency)

In [ ]:
df = customer_orders_reviews.copy()

In [ ]:
df["used_installments"] = df["payment_installments"].explode().apply(lambda x: x > 1)
df["used_installments"].sample(3)

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = (
    (
        df.explode("payment_installments")
        .explode("product_category")
        .groupby(["used_installments"])["product_category"]
        .value_counts(ascending=False)
    )[True]
    .nlargest(10)
    .plot(kind="bar")
)

ax.set_ylabel("Frequency")
ax.set_xlabel("Product Category")
ax.set_title("Installment Use by Product Category")

### Installment Use by Customer State

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = (
    (
        df.explode("payment_installments")
        .explode("product_category")
        .drop_duplicates()
        .groupby(["used_installments"])["customer_state"]
        .value_counts(ascending=False)
        / df.explode("payment_installments")
        .explode("product_category")
        .drop_duplicates()
        .groupby("customer_state")["customer_id"]
        .count()
        * 100
    )[True]
    .nlargest(10)
    .plot(kind="bar", rot=0)
)

ax.set_ylabel("Percentage of Customers")
ax.set_xlabel("Customer State")
ax.set_title("Installment Use by Customer State")

### Installments by Product Category (Average Installment Count)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7), layout="constrained")
ax = (
    round(
        (
            df.explode("payment_installments")
            .explode("product_category")
            .groupby(["product_category"])["payment_installments"]
            .mean()
        )
    )
    .nlargest(10)
    .plot(kind="bar", rot=45)
)
ax.set_ylabel("Average Number of Installments")
ax.set_xlabel("Product Category")
ax.set_title("Average Number of Installments by Product Category")

### Sale Amount by Installment Usage

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = df.groupby(["used_installments"])["full_price"].mean().plot(kind="bar", rot=0)
ax.set_ylabel("Mean Order Value")
ax.set_xlabel("")
ax.set_title("Mean Order Value: Single vs Multiple Installments")
ax.set_xticklabels(["Single Installment", "Multiple Installments"])

### Cheap vs Expensive 

In [ ]:
products = (
    df.explode("product_category").groupby(["product_category"])["full_price"].mean()
)

In [ ]:
product_categories = (
    df.explode("product_category").explode("price_category").drop_duplicates()
)
product_categories.shape

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = (
    (
        product_categories.explode("price_category")
        .explode("product_category")
        .drop_duplicates()
        .groupby("price_category")["product_category"]
        .count()
        / product_categories.shape[0]
    )
    * 100
).plot(kind="bar", rot=0)


ax.set_xlabel("Price Category")
ax.set_ylabel("Percentage of Products")
ax.set_title("Percentage of Products in Each Price Category")

### Cancelled Orders

In [ ]:
cancel_other = (
    df[df["used_installments"] == False]["order_status"].value_counts(normalize=True)
    * 100
)["canceled"]

In [ ]:
cancel_installations = (
    df[df["used_installments"] == True]["order_status"].value_counts(normalize=True)
    * 100
)["canceled"]

In [ ]:
fig, ax = plt.subplots(layout="constrained")

bars = ax.bar(
    ["With Installments", "Without Installments"], [cancel_installations, cancel_other]
)

ax.bar_label(bars, fmt="{:.2%}")
ax.set_yticks(np.arange(0, 1.1, 0.1))
ax.set_yticklabels(range(0, 101, 10))
ax.set_ylim(top=0.5)

ax.set_ylabel("Percent Cancelled")
ax.set_title("Percentage of Cancelled Orders Based on Payment Plan")

### Freight Value Reasonable?

Freight is 20.88% of the (product + freight) price


In [ ]:
(df["total_freight"] / df["full_price"]).mean() * 100

Freight is 30.84% of Product Price

In [ ]:
(df["total_freight"] / df["total_price"]).mean() * 100

### Mean Price With Payment Method

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = (
    df.explode("full_price")
    .explode("payment_type")
    .drop_duplicates()
    .groupby("payment_type")["total_price"]
    .aggregate("mean")
    .plot(kind="bar", rot=0)
)
ax.set_ylabel("Mean Sale Price")
ax.set_xlabel("Payment Method")
ax.set_title("How Mean Sale Price Changes With Payment Method")

### Installments Across Price Ranges

In [ ]:
df[df["full_price"] > 2000].count().iloc[0]

# There are only 206 orders where the full price is > 2000

In [ ]:
bins = list(range(0, 2001, 100))
labels = [f"{val-100}-{val}" for val in bins[1:]]
labels.append(">2000")
bins.append(np.inf)

df["payment_range"] = pd.cut(
    df["full_price"], bins=bins, labels=labels, include_lowest=True
)

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = (
    df.explode("payment_installments")
    .drop_duplicates()
    .groupby(["payment_range"])["payment_installments"]
    .aggregate(lambda x: np.ceil(x.mean()))
    .sort_index()
    .plot(kind="bar")
)
ax.set_xlabel("Average Order Value")
ax.set_ylabel("Average Number of Installments")
ax.set_title("Average Number of Installments Across Different Price Ranges")

### Distribution of Installment Counts

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = (df["payment_installments"].value_counts(normalize=True).sort_index() * 100).plot(
    kind="bar"
)
ax.set_ylabel("Percentage of Total Purchases")
ax.set_xlabel("Number of Installments Used")
ax.set_title("Installment Count Frequency Across Total Purchases")

### Voucher State Distribution

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = (
    df.explode("payment_type")[df["payment_type"].explode() == "voucher"]
    .drop_duplicates()["customer_state"]
    .value_counts(normalize=True)
    * 100
).plot(kind="bar")
ax.set_ylabel("Percentage of Voucher Use")
ax.set_xlabel("State")
ax.set_title("Distribution of Vouchers Among States")

### Cancelled Orders By Region

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax = (
    (
        (
            df[df["order_status"] == "canceled"]["customer_state"].value_counts()
            / df["customer_state"].value_counts()
        )
        * 100
    )
    .sort_values(ascending=False)
    .plot(kind="bar")
)
ax.set_ylabel("Percentage of State's Total Orders")
ax.set_xlabel("State")
ax.set_title(
    "Distribution of Cancelled Orders as a Proportion of Each State's Total Orders"
)